In [ ]:
import json
from pathlib import Path

import pandas as pd

In [ ]:
dbutils.widgets.text("raw_root", "/Volumes/workspace/raw_weather/clima_pe", "Raw Root")
raw_root = Path(dbutils.widgets.get("raw_root"))

In [ ]:
%sql
DROP DATABASE IF EXISTS workspace.bronze_weather CASCADE;

In [ ]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.bronze_weather
COMMENT 'Capa Bronze: clima crudo de Open-Meteo procesado' 

In [ ]:
# Un archivo por ubicacion/data_type/fecha_de_corrida; cada uno trae los
# bloques "hourly" y "daily" de la respuesta de Open-Meteo tal cual llegaron.
def load_block(block_name: str) -> pd.DataFrame:
    frames = []
    for data_type_dir in raw_root.iterdir():
        data_type = data_type_dir.name
        for json_file in data_type_dir.glob("*/*.json"):
            payload = json.loads(json_file.read_text(encoding="utf-8"))
            block = pd.DataFrame(payload[block_name])
            block["location_id"] = payload["location_id"]
            block["region"] = payload["region"]
            block["latitude"] = payload["latitude"]
            block["longitude"] = payload["longitude"]
            block["elevation"] = payload["elevation"]
            block["data_type"] = data_type
            frames.append(block)
    return pd.concat(frames, ignore_index=True)

In [ ]:
hourly = load_block("hourly")
hourly["ingested_at"] = pd.Timestamp.utcnow()
df_spark = spark.createDataFrame(hourly)

# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.bronze_weather.weather_hourly")

In [ ]:
daily = load_block("daily")
daily["ingested_at"] = pd.Timestamp.utcnow()
df_spark = spark.createDataFrame(daily)

df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.bronze_weather.weather_daily")